# CollectAddress

An end-to-end interactive test of the `CollectAddress` instruction against a real OpenAI model.

In [18]:
import sys, os

repo_root = os.path.abspath(os.path.join(os.getcwd(), "..", ".."))
if repo_root not in sys.path:
    sys.path.insert(0, repo_root)

from prompt_ml.backend.openai_backend import OpenAIBackend
from prompt_ml.fields import Field
from prompt_ml.instructions.data_collection import DataCollectionInstruction

API_KEY = "sk-proj-fw6fRj9ITkOgfYwn8O6RT3BlbkFJKZMBX9zNDq2nMmSd82Lp"

backend = OpenAIBackend(api_key=API_KEY, model="gpt-4o-mini", temperature=0.9)
print(f"Backend ready: {backend.model}")

Backend ready: gpt-4o-mini


In [19]:

class DealershipInstruction(DataCollectionInstruction):
    system_context = (
        "You are a friendly phone assistant for Acme Auto Dealership. "
        "Keep responses concise — the caller is on the phone. "
        "Speak naturally and confirm what you heard when helpful."
    )


class CollectAddress(DealershipInstruction):
    """Collect a caller's full mailing address."""

    line1:    str = Field(required=True,  description="Street address line 1")
    line2:    str = Field(required=False, description="Apartment, suite, or unit number")
    city:     str = Field(required=True,  description="City name")
    state:    str = Field(required=True,  description="State (two-letter abbreviation)")
    zip_code: str = Field(required=True,  description="Five-digit ZIP code")

    def on_complete(self) -> str:
        parts = [self.line1]
        if self.line2:
            parts.append(self.line2)
        parts += [self.city, f"{self.state} {self.zip_code}"]
        addr = ", ".join(parts)
        return f"Perfect, I have your address as {addr}. Is that correct?"


instr = CollectAddress(backend=backend)
print("Instruction ready. Missing fields:", instr.validate())

Instruction ready. Missing fields: ['line1', 'city', 'state', 'zip_code']


In [20]:

opening = instr._ask_for(instr.validate()[0])
instr._history.append({"role": "assistant", "content": opening})
print(f"\nAssistant: {opening}")


Assistant: Great! Let’s get started. Can you please provide me with your street address line 1?


In [21]:
user_input = "I live at 742 Evergreen Terrace"
# user_input = "Springfield, Illinois"
# user_input = "56010"
# user_input = "yes"

print(f"You:       {user_input}")
reply = instr.execute(user_input)
print(f"Assistant: {reply}")

if instr.is_complete():
    print("\n✓ All required fields collected.")

You:       I live at 742 Evergreen Terrace
Assistant: Thanks for that! Could you tell me the city where you live?


In [12]:
print("=== Collected fields ===")
for name, value in instr.collected().items():
    status = "✓" if value is not None else "✗"
    print(f"  {status} {name}: {value!r}")

print()
missing = instr.validate()
if missing:
    print("Still needed:", missing)
else:
    print("Complete!", instr.is_complete())

=== Collected fields ===
  ✓ line1: '742 Evergreen Terrace'
  ✗ line2: None
  ✓ city: 'Springfield'
  ✓ state: 'IL'
  ✓ zip_code: '56010'

Complete! False


In [13]:
print("=== Conversation history ===")
for i, msg in enumerate(instr.history):
    role = msg["role"].upper()
    print(f"[{i}] {role}: {msg['content']}")
    print()

=== Conversation history ===
[0] ASSISTANT: Great! Can I have your street address line 1, please?

[1] USER: 56010

[2] ASSISTANT: It looks like you provided a ZIP code. Could you give me your street address line 1?

[3] USER: I live at 742 Evergreen Terrace

[4] ASSISTANT: Thanks for confirming your address! Now, could you please tell me the city name where you live?

[5] USER: Springfield, Illinois

[6] ASSISTANT: Got it, Springfield, Illinois! Now, could you provide me with your five-digit ZIP code?

[7] USER: 56010

[8] ASSISTANT: Perfect! Just to summarize, your address is 742 Evergreen Terrace, Springfield, IL, 56010. Can you please confirm that is correct?

[9] USER: yes

[10] ASSISTANT: Perfect, I have your address as 742 Evergreen Terrace, Springfield, IL 56010. Is that correct?

[11] USER: yes

[12] ASSISTANT: Thank you for confirming! How can I assist you further today?



In [14]:
instr.reset()
print("Instruction reset. State:", instr.collected())
print("History cleared:", instr.history)

Instruction reset. State: {'line1': None, 'line2': None, 'city': None, 'state': None, 'zip_code': None}
History cleared: []


In [15]:
instr.collected()

{'line1': None, 'line2': None, 'city': None, 'state': None, 'zip_code': None}